In [19]:
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q pillow

In [20]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from datetime import datetime
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
from PIL import Image
import glob
from tensorflow import keras
from tensorflow.keras import layers, Model
from keras.losses import BinaryCrossentropy

In [21]:
BASEPATH = '/kaggle/input/datasets/syedzaidalii/deepfake/df 40'
IMG_SIZE = 224
BATCH_SIZE = 16  
EPOCHS = 10 
AUTOTUNE = tf.data.AUTOTUNE

INITIAL_LR = 2e-4  
WARMUP_EPOCHS = 2

In [22]:
def create_augmentation_layer(is_training=True):
    """Minimal augmentation to allow model to memorize training data"""
    if not is_training:
        return keras.Sequential([layers.Rescaling(1./255)])
    
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.Rescaling(1./255),
    ], name="augmentation")

In [23]:
def create_dataset(data_dir, subset, img_size=224, batch_size=8, is_training=True):
    subset_dir = os.path.join(data_dir, subset)
    
    dataset = keras.preprocessing.image_dataset_from_directory(
        subset_dir,
        labels='inferred',
        label_mode='binary',
        class_names=['fake images', 'real images'],
        color_mode='rgb',
        batch_size=batch_size,
        image_size=(img_size, img_size),
        shuffle=is_training,
        seed=42
    )
    
    augmentation = create_augmentation_layer(is_training)
    dataset = dataset.map(
        lambda x, y: (augmentation(x, training=is_training), y),
        num_parallel_calls=AUTOTUNE
    )
    
    if is_training:
        dataset = dataset.shuffle(500)  # Smaller buffer
        dataset = dataset.repeat()
    
    dataset = dataset.prefetch(buffer_size=AUTOTUNE)
    return dataset

In [24]:
def build_overfit_model(img_size=224):
  
    inputs = keras.Input(shape=(img_size, img_size, 3), name='input_image')
    

    backbone = keras.applications.EfficientNetV2B3(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )
    backbone.trainable = False
    
    x = backbone(inputs)
    
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.BatchNormalization(name='bn1')(x)  # NEW - stabilizes training
    x = layers.Dropout(0.5, name='dropout1')(x)  # INCREASED from 0.3
    
    x = layers.Dense(256, activation='relu', name='dense256')(x)
    x = layers.BatchNormalization(name='bn2')(x)  # NEW
    x = layers.Dropout(0.3, name='dropout2')(x)
    
    # Output
    outputs = layers.Dense(
        1, 
        activation='sigmoid',
        dtype='float32',
        name='output'
    )(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='DeepfakeDetector')
    print(f"Model built: {model.count_params():,} parameters")
    
    return model

In [25]:
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, warmup_steps, total_steps):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.warmup_lr = initial_lr * 0.1
        
    def __call__(self, step):
        
        if step < self.warmup_steps:
            return self.warmup_lr + (self.initial_lr - self.warmup_lr) * (
                step / self.warmup_steps
            )
        
        progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        cosine_decay = 0.5 * (1 + tf.cos(tf.constant(np.pi) * progress))
        return self.initial_lr * cosine_decay

In [26]:
def compile_model(model, steps_per_epoch, epochs):
    total_steps = steps_per_epoch * epochs
    warmup_steps = steps_per_epoch * WARMUP_EPOCHS
    
    lr_schedule = WarmupCosineDecay(INITIAL_LR, warmup_steps, total_steps)
    
    optimizer = keras.optimizers.Adam(
        learning_rate=lr_schedule,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7,
        clipnorm=5.0  # Gradient clipping
    )
    
    optimizer = keras.mixed_precision.LossScaleOptimizer(optimizer)
    
    loss = keras.losses.BinaryCrossentropy(from_logits=False)
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc'),
        ]
    )
    
    print("Model compiled for maximum training accuracy")
    print(f"  - Initial LR: {INITIAL_LR}")
    print(f"  - NO dropout, NO label smoothing = overfitting allowed")
    
    return model

In [28]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

keras.mixed_precision.set_global_policy('float32')

# Create datasets
train_ds = create_dataset(BASEPATH, 'train', IMG_SIZE, BATCH_SIZE, is_training=True)
val_ds = create_dataset(BASEPATH, 'val', IMG_SIZE, BATCH_SIZE, is_training=False)

train_size = 6364
val_size = 3195
steps_per_epoch = train_size // BATCH_SIZE
validation_steps = val_size // BATCH_SIZE

print(f"Training: {train_size} images, {steps_per_epoch} steps/epoch")

# Build model - MODIFY OUTPUT LAYER
def build_model_fixed(img_size=224):
    inputs = keras.Input(shape=(img_size, img_size, 3))
    
    backbone = keras.applications.EfficientNetV2B3(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )


    x = backbone(inputs)
    
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.BatchNormalization(name='bn1')(x)  
    x = layers.Dropout(0.5, name='dropout1')(x)  
    
    x = layers.Dense(256, activation='relu', name='dense256')(x)
    x = layers.BatchNormalization(name='bn2')(x)  # NEW
    x = layers.Dropout(0.3, name='dropout2')(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    return keras.Model(inputs=inputs, outputs=outputs)

model = build_model_fixed(IMG_SIZE)

# Simple optimizer without scheduler
optimizer = keras.optimizers.Adam(
    learning_rate=INITIAL_LR,
    clipnorm=5.0
)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall', 
             keras.metrics.AUC(name='auc')]
)

print("Model compiled (simplified)")

# Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(
        '/kaggle/working/best_model.keras',
        monitor='accuracy',
        mode='max',
        save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7
    ),
    keras.callbacks.CSVLogger('/kaggle/working/training_log.csv'),
]

# TRAIN
history = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f"\nTraining completed!")
print(f"Best accuracy: {max(history.history['accuracy']):.4f}")


Found 6364 files belonging to 2 classes.
Found 3195 files belonging to 2 classes.
Training: 6364 images, 397 steps/epoch
Model compiled (simplified)
Epoch 1/10


I0000 00:00:1774856402.233825     179 service.cc:152] XLA service 0x7d0d9000c350 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774856402.233874     179 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774856402.233879     179 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1774856421.325397     179 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-30 07:40:39.475478: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 07:40:39.610103: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 07:40:40.431609: E external/local_xl

115/397 ━━━━━━━━━━━━━━━━━━━━ 35s 125ms/step - accuracy: 0.6250 - auc: 0.6726 - loss: 0.7708 - precision: 0.6248 - recall: 0.6470

2026-03-30 07:42:28.351982: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 07:42:28.486058: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 07:42:29.312928: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 07:42:29.448551: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 07:42:30.883165: E external/local_xla/xla/stream_

397/397 ━━━━━━━━━━━━━━━━━━━━ 364s 422ms/step - accuracy: 0.7468 - auc: 0.8215 - loss: 0.5522 - precision: 0.7505 - recall: 0.7520 - val_accuracy: 0.5374 - val_auc: 0.6957 - val_loss: 0.9538 - val_precision: 0.8363 - val_recall: 0.0901 - learning_rate: 2.0000e-04
Epoch 2/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 62s 155ms/step - accuracy: 0.9457 - auc: 0.9873 - loss: 0.1434 - precision: 0.9458 - recall: 0.9458 - val_accuracy: 0.5035 - val_auc: 0.5513 - val_loss: 1.5286 - val_precision: 1.0000 - val_recall: 0.0044 - learning_rate: 2.0000e-04
Epoch 3/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 62s 155ms/step - accuracy: 0.9751 - auc: 0.9967 - loss: 0.0680 - precision: 0.9732 - recall: 0.9781 - val_accuracy: 0.5496 - val_auc: 0.7348 - val_loss: 2.3763 - val_precision: 0.9873 - val_recall: 0.0982 - learning_rate: 2.0000e-04
Epoch 4/10
397/397 ━━━━━━━━━━━━━━━━━━━━ 61s 154ms/step - accuracy: 0.9818 - auc: 0.9976 - loss: 0.0546 - precision: 0.9809 - recall: 0.9833 - val_accuracy: 0.7447 - val_auc: 0.7999 - val_loss

In [ ]:
def save_keras_model(model, name="deepfake_EF_v1"):
    try:
        save_path = f"/kaggle/working/{name}.keras"
        model.save(save_path)
        print(f"Keras model saved successfully at: {save_path}")
    except Exception as e:
        print(f"Error saving Keras model: {e}")
